In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

import xgboost as xgb

In [2]:
dfc = pd.read_csv(r'D:\Emi Predict\Data Sets\emi_prediction_cleanafterFE.csv')

features = [
    "monthly_salary","school_fees","college_fees","travel_expenses",
    "groceries_utilities","other_monthly_expenses",
    "current_emi_amount","credit_score","requested_amount","bank_balance","emergency_fund"
]

X = dfc[features]

In [3]:
y = dfc['max_monthly_emi']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

XGB Boost Regression

In [4]:
import xgboost as xgb
import pickle
xgb_model  = xgb.XGBRegressor(
        n_estimators=50,      # reduced from 200 → 50
        learning_rate=0.1,
        max_depth=3,          # smaller depth for speed
        random_state=42,
        n_jobs=-1             # parallel computation
    )
xgb_model.fit(X_train, y_train)
pickle.dump(xgb_model, open(r"D:\Emi Predict\Models-1\reg_xgb_1.pkl", "wb"))
print("Regression model saved")

Regression model saved


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

📊 Regression Metrics
RMSE : 0.0280
MAE  : 0.0218
R2   : 0.6732
MAPE : 1.90%


Linear regression

In [6]:
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (scaling + linear regression)
pipeline_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('linreg', LinearRegression())
])

# Train model
pipeline_linear.fit(X_train, y_train)

# Predictions
y_pred_linear = pipeline_linear.predict(X_test)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
mae = mean_absolute_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

# Avoid division by zero in MAPE
mape = np.mean(np.abs((y_test - y_pred_linear) / (y_test + 1e-8))) * 100

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(pipeline_linear, open(r"D:\Emi Predict\Models-1\reg_linear_1.pkl", "wb"))

📊 Regression Metrics
RMSE : 0.0311
MAE  : 0.0244
R2   : 0.5963
MAPE : 2.13%


In [ ]:
import numpy as np
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline
pipeline_lin = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('model', Ridge())   # placeholder (will be replaced in grid)
])

# Parameter grid (Ridge, Lasso, ElasticNet)
param_grid = [
    {
        'poly__degree': [1, 2, 3],
        'model': [Ridge()],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'poly__degree': [1, 2, 3],
        'model': [Lasso(max_iter=5000)],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10]
    },
    {
        'poly__degree': [1, 2, 3],
        'model': [ElasticNet(max_iter=5000)],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10],
        'model__l1_ratio': [0.1, 0.5, 0.9]
    }
]

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search
grid = GridSearchCV(
    pipeline_lin,
    param_grid,
    cv=cv,
    scoring='r2',   # optimize for R²
    n_jobs=-1,
    verbose=1
)

# Train
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predictions
y_pred_lin1 = best_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lin1))
mae = mean_absolute_error(y_test, y_pred_lin1)
r2 = r2_score(y_test, y_pred_lin1)
mape = np.mean(np.abs((y_test - y_pred_lin1) / (y_test + 1e-8))) * 100

print("Best Parameters:", grid.best_params_)

print("\n📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(best_model, open(r"D:\Emi Predict\Models-1\linear_tuned_1.pkl", "wb"))

Fitting 5 folds for each of 78 candidates, totalling 390 fits


RANDOM FOREST REGRESSOR

In [7]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (scaling + random forest)
pipeline_RF = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

# Train model
pipeline_RF.fit(X_train, y_train)

# Predictions
y_pred_RF = pipeline_RF.predict(X_test)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_RF))
mae = mean_absolute_error(y_test, y_pred_RF)
r2 = r2_score(y_test, y_pred_RF)

# Avoid division by zero in MAPE
mape = np.mean(np.abs((y_test - y_pred_RF) / (y_test + 1e-8))) * 100

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
with open(r"D:\Emi Predict\Models-1\reg_RF_1.pkl", "wb") as f:
    pickle.dump(pipeline_RF, f)

📊 Regression Metrics
RMSE : 0.0252
MAE  : 0.0184
R2   : 0.7346
MAPE : 1.60%


In [ ]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (RF doesn't need scaling, but keeping structure consistent)
pipeline_rf = Pipeline([
    ('rf', RandomForestRegressor(random_state=42))
])

# Hyperparameter distribution
param_dist = {
    'rf__n_estimators': [100, 200, 300, 500],
    'rf__max_depth': [None, 10, 20, 30, 50],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None],
    'rf__bootstrap': [True, False]
}

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search (faster than GridSearch)
search = RandomizedSearchCV(
    pipeline_rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train
search.fit(X_train, y_train)

# Best model
rf_model = search.best_estimator_

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)
mape = np.mean(np.abs((y_test - y_pred_rf) / (y_test + 1e-8))) * 100

print("Best Parameters:", search.best_params_)

print("\n📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(rf_model, open(r"D:\Emi Predict\Models-1\reg_rf_tuned.pkl", "wb"))

In [ ]:
BEST MODEL

In [9]:
def evaluate_regression(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return rmse, mae, r2, mape

In [12]:
models = {
    "Linear Regression": pipeline_linear,
    "Random Forest": pipeline_RF,
    "XGBoost": xgb_model
}

results = []


for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)
print(results_df)

Training Linear Regression...
Training Random Forest...
Training XGBoost...
               Model      RMSE       MAE        R2
0  Linear Regression  0.031075  0.024426  0.596325
1      Random Forest  0.025199  0.018361  0.734551
2            XGBoost  0.027961  0.021804  0.673186


In [13]:
results_df = results_df.sort_values(by="R2", ascending=False)
print("Model Comparison:")
print(results_df)

Model Comparison:
               Model      RMSE       MAE        R2
1      Random Forest  0.025199  0.018361  0.734551
2            XGBoost  0.027961  0.021804  0.673186
0  Linear Regression  0.031075  0.024426  0.596325


In [14]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Regression Model:", best_model_name)

Best Regression Model: Random Forest
